# Iterative methods for the numerical solution of linear systems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lorenzotomada/numerical_analysis_2026_2027/blob/main/labs/lab05/iterative_methods_for_linear_systems.ipynb)

With iterative methods, the numerical solution of the linear system $Ax = b$ is obtained as the limit of a sequence of solutions $x^{(k)}$ of simpler linear systems. 

While direct methods proceed by algebraic transformations of the initial matrix, and are therefore exact modulo finite-precision arithmetic, iterative method imply necessarily a _truncation error_.
On the other hand, iterative method preserve the sparsity of the matrix, an are therefore much more efficient when dealing large sparse systems (i.e. most of the times...).


## General scheme of iterative methods

Given a matrix $A \in \mathbb{R}^{n\times n}$, we consider a decomposition of the form 
$$
A = P - (P - A) ,
$$
hence 
$$
Ax = b \iff Px = (P-A)x + b
$$
so that it makes sense to define the iterative scheme as
$$
x^{(k+1)} = B x^{(k)} + g 
$$
with $B = P^{-1}(P-A)$ and $g = P^{-1} b$, where $B$ is called *iteration matrix*.


### Conditions for the convergence of an iterative method

__Definition:__
An iterative method is said to be _convergent_ if for any initial condition $x^{(0)} \in \mathbb{R}^n$, the sequence $\{x^{(k)}\}_{k \in \mathbb{N}}$ is convergent.

__Theorem:__ 
Necessary and sufficient condition for the convergence of an iterative method is that the _spectral radius_ of the iteration matrix $B$ is strictly less than $1$: 
$$ \rho(B) < 1. $$

__Definition:__
The _spectral radius_ of a matrix $B$ is defined as 
$$
\rho(B) = \max_i\vert \lambda_i\vert,
$$
where $\lambda_i$ are the eigenvalues of $B$.

### Stopping conditions for iterative methods

Given a tolerance $\epsilon$, one can use different stopping criteria:
- Step difference: $\Vert x^{(k+1)} - x^{(k)}\Vert < \epsilon$
- Relative step difference: $\frac{\Vert x^{(k+1)} - x^{(k)}\Vert}{x^{(k)}} < \epsilon$
Having defined the _residual_ at step $k$ as $r^{(k)} = A x^{(k)} - b$,
- Residual norm: $\Vert r^{(k)} \Vert < \epsilon$
- Relative residual norm: $\frac{\Vert r^{(k)}\Vert}{\Vert b\Vert}  < \epsilon$


## Jacobi method

If the diagonal elements of $A$ are not zero, we can consider 
$$
P = D = (a_{ii})_{i=1}^{n}, 
$$
so that the corresponding iterative method becomes:
$$
D x^{(k+1)} = (D - A)x^{k} +  b,
$$
 namely:
$$
\begin{bmatrix}
a_{11} &0&\dots&0\\
0&a_{22}&\dots&0\\
0&\dots&\dots&0\\
0&\dots&\dots&a_{nn}
\end{bmatrix} x^{k+1} = \begin{bmatrix}
0 &-a_{12}&\dots&-a_{1n}\\
-a_{21}&0&\dots&-a_{2n}\\
\dots&\dots&\dots&\dots\\
-a_{n1}&\dots&\dots&0
\end{bmatrix}x^k + b.
$$
This system becomes: $x^{(k+1)} = D^{-1} (D - A)x^{k} +  D^{-1}b$.

So, in this case $B=B_J = D^{-1}(D-A)$ and $g_J=D^{-1}b$.
By components:
$$
x_i^{(k+1)} = \frac{1}{a_{ii}} \left[b_i - \sum_{\substack{j=1 \\ j\neq i}}^{n} a_{ij}x_j^{(k)}\right] 
= \frac{1}{a_{ii}} \left[b_i - \sum_{j=1}^{i-1} a_{ij}x_j^{(k)} - \sum_{j=i+1}^{n} a_{ij}x_j^{(k)}\right] ,\quad i = 1,\ldots, n,
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
A = np.arange(1, 17).reshape(4, 4)
D = np.diag(np.diag(A))

B_J = np.linalg.inv(D) @ (D - A)
B_J

We can check the spectral radius

In [ ]:
rho_B_J = max(abs(np.linalg.eigvals(B_J)))
print(rho_B_J)

This means the convergence of the Jacobi iterative method is not guaranteed. It is not difficult to see that this is true every time the system matrix $A$ is singular.

Let us now define a function to perform the Jacobi method:

In [ ]:
def jacobi(A, b, x0 = None, tol=1e-10, max_iter=100):
    """
    Solves the system of linear equations Ax = b using the Jacobi method.
    Stopping criterion based on 2-norm of residual.
    
    Parameters:
    A: Coefficient matrix (must be square).
    b: Right-hand side vector.
    tol (optional): Tolerance for the stopping criterion. Defaults to 1e-10.
    max_iter (optional): Maximum number of iterations. Defaults to 100.
    
    Returns:
    Solution vector x, iteration count.
    """
    try:    
        n, m = A.shape
    except:
        raise ValueError("Input must be a 2D array (matrix).")
    
    assert n == m, "Matrix A must be square."
    assert n == len(b), "Matrix A and vector b must have the same number of rows."

    if x0 is None:
        x = np.zeros(n)
    else:
        x = x0.copy()

    x_new = np.zeros(n)

    res = np.linalg.norm(A@x - b, 2)
    iter = 0
    res_all = []
    while iter < max_iter and res > tol:
        for i in range(n):
            x_new[i] = # complete here
        
        x = x_new.copy()
        res = np.linalg.norm(A@x - b, 2)
        iter += 1
        # collect residuals at each iteration (for plotting)
        res_all.append(res)
    return x, iter, res_all

### Example

In [ ]:
# Generate a NxN system
N = 500
A = np.random.rand(N, N) + N*np.eye(N)
x_exact = np.random.rand(N)
b = A@x_exact

# Solve the system using Jacobi
x_jac, iter_jac, res_all_jac = jacobi(A, b, tol=1e-10, max_iter=10000)

# Compare the solution with the exact solution
print("Found solution in ", iter_jac, " iterations.")
print("Relative error: ", np.linalg.norm(x_jac - x_exact, 2)/np.linalg.norm(x_exact, 2))

## Gauss-Seidel method

In this case, we choose a different preconditioner:

$$
P = D-E,
$$
where $E$ is the lower triangular matrix:
$$
\begin{cases}
E_{ij}=-a_{ij}\text{ if }i>j,\\
E_{ij}=0 \text{ if }i\leq j.
\end{cases}
$$
In this case:
$$
(D-E) x^{(k+1)} = (D - E-A)x^{k} +  b,
$$
 namely:
$$
\begin{bmatrix}
a_{11} &0&\dots&0\\
a_{21}&a_{22}&\dots&0\\
\dots&\dots&\dots&0\\
a_{n1}&\dots&\dots&a_{nn}
\end{bmatrix} x^{k+1} = \begin{bmatrix}
0 &-a_{12}&\dots&-a_{1n}\\
0&0&\dots&-a_{2n}\\
\dots&\dots&\dots&\dots\\
0&\dots&\dots&0
\end{bmatrix}x^k + b.
$$
This system becomes: $x^{(k+1)} = (D-E)^{-1} (D -E- A)x^{k} +  (D-E)^{-1}b$.

So, in this case $B_{GS} = (D-E)^{-1}(D-E-A)$ and $g_{GS}=(D-E)^{-1}b$.

Looking back at the Jacobi iteration, we see that at the $i$-th row we already have $x^{(k+1)}_{j}$ for $j=1,\ldots,i-1$, so we can use those results:
$$
x_i^{(k+1)} = \frac{1}{a_{ii}} \left[b_i - \sum_{j=1}^{i-1} a_{ij}x_j^{(k+1)} - \sum_{j=i+1}^{n} a_{ij}x_j^{(k)}\right] ,\quad i = 1,\ldots, n,
$$

In [ ]:
def gauss_seidel(A, b, tol=1e-10, max_iter=100):
    """
    Solves the system of linear equations Ax = b using the Gauss-Seidel method.
    Stopping criterion based on 2-norm of residual.
    """

    try:    
        n, m = A.shape
    except:
        raise ValueError("Input must be a 2D array (matrix).")
    
    assert n == m, "Matrix A must be square."
    assert n == len(b), "Matrix A and vector b must have the same number of rows."

    x = np.zeros(n)
    x_new = np.zeros(n)

    iter = 0
    res_all = []
    res = tol + 1
    while iter < max_iter and res > tol:
        for i in range(n):
            x_new[i] = # complete here

        x = x_new.copy()
        res = np.linalg.norm(np.dot(A, x) - b, 2)
        res_all.append(res)
        iter += 1
    return x, iter, res_all

### Example

In [ ]:
# Solve the system using Gauss-Seidel
x_gs, iter_gs, res_all_gs = gauss_seidel(A, b)

# Compare the solution with the exact solution
print("Found solution in ", iter_gs, " iterations.")
print("Relative error: ", np.linalg.norm(x_gs - x_exact, 2)/np.linalg.norm(x_exact, 2))

# Plot convergence

## Comparison between Jacobi and Gauss-Seidel

Let u s compare the spectral radius of the iteration matrices for the two methods

In [ ]:
# A = np.array([[3, 0, 4], [7, 4, 2], [-1, -1, -2]])
# A = np.random.rand(5, 5) +5* np.eye(5)
A = np.array([[-3, 3, -6], [-4, 7, -8], [5, 7, -9]])


D = np.diag(np.diag(A))
E = - np.tril(A, -1)
F = - np.triu(A, 1)
# E+F is equivalent to D-A

B_J = np.linalg.inv(D)@(D-A)
B_GS = np.linalg.inv(D-E)@(D-E-A)

print("Spectral radius of B_J: ", max(abs(np.linalg.eigvals(B_J))))
print("Spectral radius of B_GS: ", max(abs(np.linalg.eigvals(B_GS))))

__Theorem__
Let $A$ be a _tridiagonal_ matrix of order $n$ with principal elements $\neq 0$. Then Gauss-Seidel method converges $\iff$ Jacobi method converges and in particular
$$
\rho(B_\text{GS}) = \rho(B_\text{J})^2,
$$
so that in these conditions Gauss-Seidel is twice as fast.

In [ ]:
N = 100
A = 2*np.eye(N) + np.diag(-np.ones(N-1), -1) + np.diag(-np.ones(N-1), 1)

D = np.diag(np.diag(A))
E = - np.tril(A, -1)
F = - np.triu(A, 1)

B_J = np.linalg.inv(D)@(D-A)
B_GS = np.linalg.inv(D-E)@(D-E-A)

rho_J = max(abs(np.linalg.eigvals(B_J)))
rho_GS = max(abs(np.linalg.eigvals(B_GS)))

print("Spectral radius of B_J: ", rho_J)
print("Spectral radius of B_GS: ", rho_GS)

## Richardson methods

Starting again from the decomposition $A = P - (P-A)$, we can note that $Ax = b \iff Px = (P-A)x + b$, which defines the iteration $Px^{(k+1)} = (P-A)x^{(k)} + b$, so that 
$$
P(x^{(k+1)} - x^{(k)}) = r^{(k)}, 
$$
which can be further generalized introducing an acceleration parameter $\alpha_k$ at each iteration:
$$
P(x^{(k+1)} - x^{(k)}) = \alpha_k r^{(k)} \iff x^{(k+1)} = x^{(k)} +  \alpha_k P^{-1} r^{(k)}. 
$$
This is in general called preconditioned Richardson's method and it can be 
- stationary, if $\alpha_k = \alpha $; or 
- dynamic if the step-size varies at each iteration.

__Observation__ 
For $\alpha=1$, we find the previous methods by setting 
- $P = D$, we get Jacobi,
- $P = D-E$, we get Gauss-Seidel.

### Stationary preconditioned Richardson method

For the stationary Richardson method, if we start from $r^{(0)}=b-Ax^{(0)}$, the algorithm follows this step at each $k \geq 0$:
$$
\begin{split}
&z^{(k)}=P^{-1}r^{(k)}\\
&x^{(k+1)}=x^{(k)}+\alpha z^{(k)}\\
&r^{(k+1)}=r^{(k)}-\alpha A z^{(k)}.\\
\end{split}
$$

#### How do we choose $\alpha$?

The iteration matrix for the stationary preconditioned Richardson method is 
$$
R_\alpha = I - \alpha P^{-1} A,
$$ 
and its eigenvalues are
$$
1 - \alpha \eta_i,
$$
being $\eta_i$ the eigenvalues of $P^{-1} A$.


If $A$ is symmetric and positive definite, one can see that the optima value for the acceleration parameter is
$$
\alpha^* = \frac{2}{\eta_{\max} + \eta_{\min}}, 
$$
for which 
$$
\rho(R_\alpha) = \frac{\eta_{\max} - \eta_{\min}}{\eta_{\max} + \eta_{\min}}.
$$


In [ ]:

def richardson(A, b, P=None, alpha=None, tol=1e-10, max_iter=100):
    """
    Solves the system of linear equations Ax = b using the Richardson method.
    
    Parameters:
    A: Coefficient matrix (must be symmetric and positive definite).
    b: Right-hand side vector.
    P (optional): Preconditioner matrix. Defaults to the identity matrix.
    alpha (optional): Relaxation parameter. If None, it will be set to the optimal value.
    tol (optional): Tolerance for the stopping criterion. Defaults to 1e-10.
    max_iter (optional): Maximum number of iterations. Defaults to 100.
    
    Returns:
    Solution vector x, iteration count.
    """
    
    n, m = A.shape
    assert n == m, "Matrix A must be square."
    assert n == len(b), "Matrix A and vector b must have the same number of rows."
    assert np.allclose(A, A.T), "Matrix A must be symmetric."
    
    if P is None:
        P = np.eye(n)
    
    if alpha is None:
        eigvals = np.linalg.eigvals(np.linalg.inv(P) @ A)
        alpha = 2 / (np.max(eigvals) + np.min(eigvals))
    
    x = np.zeros(n)
    r = b - A @ x
    
    iter = 0
    res = tol + 1
    res_all = []

    while iter < max_iter and res > tol:
        z = # complete here
        x = # complete here
        r = # complete here
        res = np.linalg.norm(r, 2)
        iter += 1
        res_all.append(res)
    return x, iter, res_all

### Preconditioned gradient method

We can chose $\alpha_k$ so to minimize the square of the error at step $k+1$ in the norm induced by $A$, which we denote by 
$$
\Vert e^{(k+1)} \Vert_A ^2 = (e^{(k+1)}, e^{(k+1)})_A =  (A e^{(k+1)}, e^{(k+1)}).
$$
Noting that 
$$
A (x^* - x^{(k+1)}) = b - A x^{(k+1)} = r^{(k+1)},
$$
and we can look for the optimal $\alpha_k$ by stationarity of $(r^{(k+1)}, e^{(k+1)})$, which turns out to be 
$$
\alpha_k = \frac{(z^{(k)}, r^{(k)})}{(A z^{(k)}, z^{(k)})},
$$
where $z^{(k)} = P^{-1} r^{(k)}$ is the preconditioned residual. 

So, we start from $r^{(0)}=b-Ax^{(0)}$ the algorithm follows this step at each $k \geq 0$:
$$
\begin{split}
&z^{(k)}=P^{-1}r^{(k)}\\
&\alpha_k = \dfrac{(r^{(k)}, z^{(k)})}{(Az^{(k)}, z^{(k)})}\\
&x^{(k+1)}=x^{(k)}+\alpha_k z^{(k)}\\
&r^{(k+1)}=r^{(k)}-\alpha_k A z^{(k)}\\
\end{split}
$$

In [ ]:
def gradient(A, b, x0 = None, P=None, alpha=None, tol=1e-10, max_iter=100, history=False):
    """
    Solves the system of linear equations Ax = b using the gradient method.
    
    Parameters:
    A: Coefficient matrix (must be symmetric and positive definite).
    b: Right-hand side vector.
    P (optional): Preconditioner matrix. Defaults to the identity matrix.
    tol (optional): Tolerance for the stopping criterion. Defaults to 1e-10.
    max_iter (optional): Maximum number of iterations. Defaults to 100.
    
    Returns:
    Solution vector x, iteration count.
    """
    
    n, m = A.shape
    assert n == m, "Matrix A must be square."
    assert n == len(b), "Matrix A and vector b must have the same number of rows."
    assert np.allclose(A, A.T), "Matrix A must be symmetric."
    
    if x0 is None:
        x = np.zeros(n)
    else:
        x = x0.copy()

    if history:
        x_hist = [x.copy()]

    if P is None:
        P = np.eye(n)
    
    r = b - A @ x
    
    iter = 0
    res = tol + 1
    res_all = []
    while iter < max_iter and res > tol:
        z = # complete here
        alpha = # complete here
        x = # complete here
        r = # complete here
        res = np.linalg.norm(r, 2)
        iter += 1
        res_all.append(res)
    
        if history:
            x_hist.append(x.copy())
    
    if history:
        return np.array(x_hist)

    return x, iter, res_all


## Conjugate Gradient (CG)

In the gradient method, step $k+1$ is taken in the direction of $z^{(k)}$ and the step has length $\alpha_k$ that minimizes $\Vert e^{(k+1)}\Vert_A^2$.
In the conjugate gradient method, the idea is to take the $k+1$-th step in a direction $p^{(k+1)} = z^{(k+1)}-\beta_k p^{(k)}$, where the parameter $\beta_k$ is chosen such that the new direction is A-orthogonal with respect to the previous direction:
$$(A p^{(k)},p^{(k+1)}) = 0,$$
meaning that two subsequent descent directions are orthogonal in the scalar product induced by $A$. In other words successive steps are mutually independent in the sense of the A-product. 

So, we start from $r^{(0)}=b-Ax^{(0)}$, $z^{(0)}=P^{-1}r^{(0)}$ , $p^{(0)}=z^{(0)}$. The algorithm follows this step at each $k \geq 0$:
$$
\begin{aligned}
&\alpha_k = \dfrac{(z^{(k)}, r^{(k)})}{(A p^{(k)}, p^{(k)})}\\
&x^{(k+1)}=x^{(k)}+\alpha_k p^{(k)}\\
&r^{(k+1)}=r^{(k)}-\alpha_k A p^{(k)}\\
&z^{(k+1)}=P^{-1}r^{(k+1)}\\
&\beta_k = \dfrac{(z^{(k+1)}, Ap^{(k)})}{(p^{(k)}, Ap^{(k)})}\\
&p^{(k+1)}=z^{(k+1)}+\beta_k p^{(k)}
\end{aligned}
$$
We simplify a bit the case, and consider $P=I$ (no preconditioning). In this case, we can derive:
$$
\alpha_k = \frac{\Vert r^{(k)} \Vert^2}{(A p^{(k)}, p^{(k)})},\quad 
\beta_k = \frac{\Vert r^{(k+1)} \Vert^2}{\Vert r^{(k)} \Vert^2}
$$

In [ ]:
def conj_gradient(A, b, x0 = None, tol=1e-10, max_iter=100, history=False):
    """
    Solves the system of linear equations Ax = b using the Conjugate Gradient method.
    
    Parameters:
    A: Coefficient matrix (must be symmetric and positive definite).
    b: Right-hand side vector.
    P (optional): Preconditioner matrix. Defaults to the identity matrix.
    tol (optional): Tolerance for the stopping criterion. Defaults to 1e-10.
    max_iter (optional): Maximum number of iterations. Defaults to 100.
    
    Returns:
    Solution vector x, iteration count.
    """
    
    n, m = A.shape
    assert n == m, "Matrix A must be square."
    assert n == len(b), "Matrix A and vector b must have the same number of rows."
    assert np.allclose(A, A.T), "Matrix A must be symmetric."
    
    if x0 is None:
        x = np.zeros(n)
    else:
        x = x0.copy()

    if history:
        x_hist = [x.copy()]    

    r = b - A @ x
    p = r.copy()
    
    iter = 0
    res = tol + 1
    res_all = []
    while iter < max_iter and res > tol:
        rho = np.dot(r,r)
        Ap = A @ p
        alpha =  rho / np.dot(Ap,p)

        x += alpha * p
        r -= alpha * Ap

        rho_new = np.dot(r, r)
        beta = # complete here
        rho = rho_new
        p = # complete here

        res = np.linalg.norm(r, 2)
        iter += 1
        res_all.append(res)

        if history:
            x_hist.append(x.copy())

    if history:
        return np.array(x_hist)
    
    return x, iter, res_all

## Complete comparison

In [ ]:
# define discretization
n = 30
h = 1/(n-1)

x = np.linspace(0,1,n)

# System matrix A
a = -np.ones((n-1))
b = 2*np.ones((n))
A = (1/h**2)*(np.diag(a,-1)+np.diag(b)+np.diag(a,1))

# Boundary conditions
A[0,:] = 0
A[:,0] = 0
A[0,0] = 1

A[-1,:] = 0
A[:,-1] = 0
A[-1,-1] = 1

# Forcing term
# f = x*(1.-x)

# Exact solution
u_ex = (x**4)/12. - (x**3)/6. + x/12.
# u_ex = np.linalg.solve(A,f)
 
f = A@u_ex

In [ ]:
tol = 1e-10
max_iter = 10000
funcs = [jacobi, gauss_seidel, richardson, gradient, conj_gradient]
labels = ["Jacobi", "Gauss-Seidel", r"Richardson (P=I, best $\alpha$)", "Gradient", "Conjugate gradient"]
styles = ["b", "r", "g", "y", "k"]

for i, f in enumerate(funcs):
    sol_f = f(A, b, tol=tol, max_iter=max_iter)
    all_residuals = sol_f[-1]
    plt.semilogy(range(len(all_residuals)), sol_f[-1], styles[i], label=labels[i], markersize=1)
plt.grid()
plt.legend()
plt.show()

## Geometric view

Since we are mostly dealing with symmetric and positive definite matrices, we can interpret the solution of the linear systems as the minimazion of the quadratic form induced by the matrix
$$
F(x) = \frac{1}{2} x^T A x - x^T b.
$$

For the 2D case this can be visualized by contour plots

In [ ]:
A = np.array([[5,1],[1,1]])
x_exact = np.array([np.pi,np.pi])
b = A@x_exact

def F(x):
    return 0.5 * np.dot(x,A@x) - np.dot(x,b)

x = np.linspace(2.5, 5, 400)
y = np.linspace(-1.5, 4.5, 400)
X, Y = np.meshgrid(x, y)
Z = np.array([F(np.array([x, y])) for x, y in zip(X.flatten(), Y.flatten())]).reshape(X.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contour(X, Y, Z, levels=50)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Contour plot of $F(x)$')
ax.set_aspect('equal', 'box')

max_steps = 10
x = np.array([3., -1.])

steps_gradient = gradient(A, b, x0=x, tol=1e-10, max_iter=max_steps, history=True)
steps_conj_gradient = conj_gradient(A, b, x0=x, tol=1e-10, max_iter=max_steps, history=True)

ax.plot(steps_gradient[:,0], steps_gradient[:,1], 'mo-')
ax.plot(steps_conj_gradient[:,0], steps_conj_gradient[:,1], 'ro-')
plt.show()